# SignFlow — treinar modelo Libras no Colab (GPU grátis)

Este notebook baixa o dataset **MINDS-Libras** direto nos servidores do Google, extrai landmarks com MediaPipe, treina um Bi-LSTM na GPU T4 do Colab, e te devolve os dois arquivos (`libras_lstm.keras` + `libras_labels.json`) pra você colocar em `artifacts/` no projeto local.

Nada é baixado na sua máquina até o final — só o modelo treinado (~poucos MB).

---

## Antes de rodar

1. Menu **Ambiente de execução → Alterar tipo de ambiente de execução** → *Acelerador de hardware*: **GPU (T4)** → Salvar.
2. Ajuste `SIGNERS` na próxima célula (padrão: 4 sinalizadores).
3. Menu **Ambiente de execução → Executar tudo**.

Tempo total esperado: **15–25 min**.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# Configuração
# ─────────────────────────────────────────────────────────────────────────

# Quais sinalizadores baixar do MINDS-Libras. 1..12 disponíveis.
# Mais sinalizadores = mais variedade de estilo = modelo generaliza melhor,
# mas download e processamento mais longos.
SIGNERS = [1, 2, 3, 4]

# Branch / commit do repo a clonar. master é o default.
GITHUB_REPO = "https://github.com/ccarloshenri/realtime-sign-translator.git"
GITHUB_BRANCH = "master"

# Número de épocas de treino. O early stopping vai cortar antes se estabilizar.
EPOCHS = 60

## 1. Clonar o repo e instalar dependências

In [ ]:
!git clone --depth 1 --branch {GITHUB_BRANCH} {GITHUB_REPO} /content/signflow
%cd /content/signflow

# opencv-python-headless evita deps gráficas que a gente não precisa no Colab.
# TensorFlow já vem pré-instalado no Colab, mas garantimos uma versão recente.
!pip install -q \
    mediapipe \
    opencv-python-headless \
    'numpy>=1.24,<2.1' \
    Pillow \
    PyYAML \
    'pydantic>=2.6'

In [ ]:
# Verifica GPU
import tensorflow as tf
gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow:", tf.__version__)
print("GPUs:", gpus if gpus else "nenhuma (vai rodar em CPU, mais lento)")

## 2. Baixar o modelo MediaPipe hand_landmarker

O extrator de landmarks que o pipeline usa tanto no treino quanto em runtime.

In [ ]:
!mkdir -p artifacts
!wget -q --show-progress -O artifacts/hand_landmarker.task \
    https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task
!ls -la artifacts/

## 3. Baixar MINDS-Libras do Zenodo

Cada zip é 2.5–8 GB. O Colab tem ~100 GB de disco efêmero e conexão rápida com o Zenodo, então isso costuma levar poucos minutos por sinalizador.

In [ ]:
import os, pathlib

raw_dir = pathlib.Path("data/minds-libras/raw")
raw_dir.mkdir(parents=True, exist_ok=True)

for signer in SIGNERS:
    fname = f"Sinalizador{signer:02d}.zip"
    out = raw_dir / fname
    url = f"https://zenodo.org/records/2667329/files/{fname}"
    if out.exists() and out.stat().st_size > 0:
        print(f"[skip] {out} ({out.stat().st_size / 1e9:.2f} GB)")
        continue
    print(f"[download] {fname}")
    !wget -q --show-progress -O {out} {url}

## 4. Descompactar os vídeos

In [ ]:
import zipfile

videos_dir = pathlib.Path("data/minds-libras/videos")
videos_dir.mkdir(parents=True, exist_ok=True)

for signer in SIGNERS:
    zp = raw_dir / f"Sinalizador{signer:02d}.zip"
    print(f"[unzip] {zp.name}")
    with zipfile.ZipFile(zp) as zf:
        zf.extractall(videos_dir)

print("done")

## 5. Extrair landmarks com MediaPipe

Para cada vídeo, pegamos uma janela central de 30 frames (definido em `config.yaml`), rodamos o `MediaPipeHandLandmarkExtractor` e salvamos o tensor normalizado em `samples/<sinal>/*.npz`.

É exatamente o que o script `fetch_minds_libras.py` faz — aqui só pulamos as fases de download/unzip que já rodamos manualmente acima.

In [ ]:
signers_arg = " ".join(str(s) for s in SIGNERS)
!python -m training.libras.fetch_minds_libras \
    --signers {signers_arg} \
    --skip-download --skip-unzip \
    --cleanup-videos

In [ ]:
# Quantas amostras por classe?
import os
counts = {}
for name in sorted(os.listdir("samples")):
    p = os.path.join("samples", name)
    if os.path.isdir(p):
        counts[name] = len(os.listdir(p))
print(f"{len(counts)} classes, total = {sum(counts.values())} amostras")
for k, v in counts.items():
    print(f"  {k:14s} {v}")

## 6. Construir o dataset (train / val / test)

In [ ]:
!python -m training.preprocessing.build_dataset
!ls -la training/datasets/ artifacts/libras_labels.json 2>&1 | tail -n 10

## 7. Treinar o Bi-LSTM na GPU

Usa o `LibrasFeatureExtractor` pra enriquecer `(T, 126)` → `(T, 252)` com canais de velocidade. `--allow-unknown-labels` libera o vocabulário MINDS (diferente do `LIBRAS_BASE_VOCABULARY` seed).

In [ ]:
!python -m training.libras.train_libras_model \
    --allow-unknown-labels \
    --epochs {EPOCHS}

## 8. Avaliar no split de teste

In [ ]:
# Avaliação manual (o evaluate_model.py espera o dataset não-enriquecido, então
# aplicamos aqui o mesmo LibrasFeatureExtractor do treino)
import numpy as np
from tensorflow.keras.models import load_model
from src.implementations.libras.libras_feature_extractor import LibrasFeatureExtractor

data = np.load("training/datasets/signflow.npz", allow_pickle=True)
X_test = data["X_test"]
y_test = data["y_test"]
labels = list(data["labels"])

extractor = LibrasFeatureExtractor(include_both_hands=True)
X_test_enr = np.stack([extractor.enrich(x) for x in X_test], axis=0)

model = load_model("artifacts/libras_lstm.keras")
probs = model.predict(X_test_enr, verbose=0)
preds = np.argmax(probs, axis=1)

top1 = float(np.mean(preds == y_test))
print(f"Top-1 accuracy no teste: {top1*100:.2f}%  ({len(y_test)} amostras)")

# Confusion matrix compacta
conf = np.zeros((len(labels), len(labels)), dtype=int)
for t, p in zip(y_test, preds):
    conf[t, p] += 1
print("\nPer-class:")
for i, lbl in enumerate(labels):
    support = conf[i].sum()
    correct = conf[i, i]
    pct = (correct / support * 100) if support else 0
    print(f"  {lbl:14s} {correct}/{support}  ({pct:.1f}%)")

## 9. Baixar o modelo treinado

As duas próximas células disparam download dos dois arquivos pro seu computador.

In [ ]:
from google.colab import files
files.download("artifacts/libras_lstm.keras")

In [ ]:
files.download("artifacts/libras_labels.json")

## Próximos passos na sua máquina local

1. Coloque os dois arquivos baixados em `artifacts/` dentro do projeto SignFlow:
   - `artifacts/libras_lstm.keras`
   - `artifacts/libras_labels.json`

2. No `config.yaml` local, confirme:
   ```yaml
   classifier:
     backend: libras
   ```

3. Rode o app:
   ```
   python -m src.main
   ```

4. Faça um dos 20 sinais MINDS em frente à câmera:
   `Acontecer, Aluno, Amarelo, América, Aproveitar, Bala, Banco, Banheiro, Barulho, Cinco, Conhecer, Espelho, Esquina, Filho, Maçã, Medo, Ruim, Sapo, Vacina, Vontade`.

> **Expectativa realista**: o MINDS-Libras foi gravado em estúdio com chroma key, então o modelo aprende padrões muito limpos que podem não generalizar bem pra sua cozinha com luz amarela e fundo bagunçado. Sinais que envolvem movimentos grandes (*Cinco*, *Banheiro*, *Vontade*) costumam sobreviver melhor à transferência de domínio do que sinais sutis. Pra robustez, colete ~10 amostras próprias suas pra cada sinal que quiser reforçar e re-treine misturando com o MINDS.

## Referência

MINDS-Libras — Rezende et al., *Dataset for Brazilian Sign Language (Libras)*. Zenodo 2667329. CC-BY 4.0.
Se você publicar qualquer coisa que use esse modelo, cite o dataset.